# Criar overviews (`.ovr`) para GeoTIFFs no QGIS

Use este notebook quando tiles GeoTIFF grandes estiverem travando o QGIS ao abrir, dar zoom ou ligar/desligar camadas.

As overviews (`.ovr`) sao piramides de visualizacao: versoes reduzidas do raster que o QGIS usa em zoom baixo. Isso deixa a navegacao muito mais fluida.

## O que este script faz

- Procura `.tif` em `C:\\00_DATASETS_AI\\260515-piracicaba-aoi\\tiles`.
- Tambem procura `.tif` em `tiles\\bkp`.
- Para cada tile, roda `gdaladdo` dentro do container Docker do projeto `leucaena-earth-segmentation`.
- Cria um arquivo auxiliar por tile: `nome_do_tile.tif.ovr`.
- Nao altera o `.tif` original.

## Antes de rodar

Confirme que:

- Docker Desktop esta aberto.
- WSL2/Ubuntu esta funcionando.
- O projeto `leucaena-earth-segmentation` existe em `C:\\Users\\mathe\\OneDrive\\Documents\\0-GITHUB\\leucaena-earth-segmentation`.
- Os tiles estao em `C:\\00_DATASETS_AI\\260515-piracicaba-aoi\\tiles`.


## Script PowerShell

Abra o **PowerShell** e cole o bloco abaixo. Para outro dataset, altere apenas `$root`. Se o projeto mudar de lugar, altere `$repo`.


In [ ]:
$repo = "/mnt/c/Users/mathe/OneDrive/Documents/0-GITHUB/leucaena-earth-segmentation"
$root = "C:\\00_DATASETS_AI\\260515-piracicaba-aoi\\tiles"

$files = @(Get-ChildItem -Path $root -Filter *.tif -File) +
         @(Get-ChildItem -Path (Join-Path $root "bkp") -Filter *.tif -File -ErrorAction SilentlyContinue)

Write-Host "Arquivos encontrados: $($files.Count)"

$i = 0
foreach ($file in $files) {
    $i++

    $rel = $file.FullName.Substring($root.Length).TrimStart('\\') -replace '\\\\','/'
    $containerPath = "/data/rgbir/$rel"

    Write-Host "[$i/$($files.Count)] criando overview: $containerPath"

    $cmd = "cd $repo && docker compose run --rm --no-TTY segmentation gdaladdo -ro --config COMPRESS_OVERVIEW DEFLATE --config PREDICTOR_OVERVIEW 2 --config BIGTIFF_OVERVIEW IF_SAFER --config GDAL_TIFF_OVR_BLOCKSIZE 512 --config GDAL_NUM_THREADS ALL_CPUS -r average '$containerPath' 2 4 8 16 32 64"

    wsl.exe -d Ubuntu -e bash -lc $cmd

    if ($LASTEXITCODE -ne 0) {
        throw "gdaladdo failed for $containerPath"
    }
}

Write-Host "OK - overviews criados"


## Verificar se funcionou

Rode no PowerShell:


In [ ]:
Get-ChildItem "C:\\00_DATASETS_AI\\260515-piracicaba-aoi\\tiles" -Filter *.ovr
Get-ChildItem "C:\\00_DATASETS_AI\\260515-piracicaba-aoi\\tiles\\bkp" -Filter *.ovr


## Depois no QGIS

- Feche e reabra o QGIS, ou remova e adicione os rasters novamente.
- O QGIS detecta automaticamente os arquivos `.tif.ovr` ao lado dos `.tif`.
- Se ainda estiver pesado, aumente o cache em `Settings -> Options -> Rendering -> Raster cache (MB)` para `2048` ou `4096`.

## Observacoes

- Um unico `.ovr` guarda todos os niveis: `2, 4, 8, 16, 32, 64`.
- O `.ovr` ocupa espaco extra em disco, mas melhora muito a visualizacao.
- Para foto aerea/RGB/RGBN, `-r average` e uma boa escolha.
